# Анализ пользователей Netflix и подписок

**Цель исследования:** понять профиль пользователей, поведение подписок и выручку сервиса на основе данных `netflix_users_data.csv`, выделить ключевые сегменты и сформировать бизнес‑инсайты.

**Роль:** кейс для портфолио продуктового/бизнес‑аналитика (анализ подписочного онлайн‑сервиса).

**Данные:** обезличенные данные по пользователям Netflix: тип подписки, ежемесячная выручка, даты начала и последнего платежа, страна, возраст, пол, тип устройства и длительность плана.

## Цель исследования и описание данных

В этом кейсе мы проводим полный EDA по пользователям Netflix: оцениваем размер и структуру базы, демографию, устройства, тарифы, географию и динамику выручки. Результат — слайд‑дека с выводами и рекомендациями для продукта и маркетинга.

## Импорт библиотек и загрузка данных

Подключаем библиотеки анализа и визуализации, задаём единую красно‑оранжевую палитру (Netflix‑стиль), загружаем CSV и смотрим структуру таблицы.

In [ ]:
# Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Единая палитра для Netflix-кейса (красно-оранжевые оттенки)
sns.set_theme(style="whitegrid", palette="Reds")
pd.set_option("display.max_columns", 50)

# Загрузка данных (файл в текущей директории / в Colab — загрузите и укажите путь)
FILE_PATH = "netflix_users_data.csv"
df = pd.read_csv(FILE_PATH)

df.head()
df.info()

## Первичный обзор и очистка данных

Приводим имена колонок к snake_case, преобразуем даты (формат dd-mm-yy) и числовые поля, проверяем дубликаты и пропуски.

In [ ]:
# Приведение имён колонок к snake_case
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
df.columns.tolist()

In [ ]:
# Преобразование дат: в CSV формат dd-mm-yy
df["join_date"] = pd.to_datetime(df["join_date"], format="%d-%m-%y", errors="coerce")
df["last_payment_date"] = pd.to_datetime(df["last_payment_date"], format="%d-%m-%y", errors="coerce")

# Месячная выручка — числовой тип
df["monthly_revenue"] = pd.to_numeric(df["monthly_revenue"], errors="coerce")

df.dtypes

In [ ]:
# Дубликаты: по user_id и по всей строке
dup_user = df.duplicated(subset=["user_id"]).sum()
dup_all = df.duplicated().sum()
print(f"Дубликатов по user_id: {dup_user}")
print(f"Полных дубликатов строк: {dup_all}")
if dup_user > 0:
    df = df.drop_duplicates(subset=["user_id"], keep="first")
elif dup_all > 0:
    df = df.drop_duplicates()

# Доля пропусков по столбцам
missing = df.isna().mean().sort_values(ascending=False)
print("\nДоля пропусков:")
print(missing[missing > 0].round(4).to_string() if (missing > 0).any() else "Пропусков нет")

# Удаление строк с критическими пропусками (даты, выручка)
critical = ["join_date", "last_payment_date", "monthly_revenue"]
critical = [c for c in critical if c in df.columns]
df = df.dropna(subset=critical)
print(f"\nСтрок после очистки: {len(df)}")

**Вывод:** Имена колонок приведены к snake_case. Даты преобразованы в datetime (формат dd-mm-yy), monthly_revenue — в числовой тип. Дубликаты по user_id или по всей строке удалены. Пропуски оценены; строки с пропусками в критических полях (даты, выручка) удалены. Данные готовы к расчёту метрик и визуализациям.

## Базовые метрики: пользователи, период, выручка

Оцениваем размер базы, период наблюдения и масштаб месячной выручки.

In [ ]:
n_users = df["user_id"].nunique()
join_min, join_max = df["join_date"].min(), df["join_date"].max()
pay_min, pay_max = df["last_payment_date"].min(), df["last_payment_date"].max()
rev_mean = df["monthly_revenue"].mean()
rev_median = df["monthly_revenue"].median()
total_mrr = df["monthly_revenue"].sum()

print(f"Уникальных пользователей:     {n_users:,}")
print(f"Период join_date:             {join_min.date()} — {join_max.date()}")
print(f"Период last_payment_date:    {pay_min.date()} — {pay_max.date()}")
print(f"Средняя месячная выручка:   {rev_mean:.2f}")
print(f"Медианная месячная выручка: {rev_median:.2f}")
print(f"Суммарная MRR (все пользователи): {total_mrr:,.0f}")
print("\nРаспределение по типам подписки:")
print(df["subscription_type"].value_counts().to_string())

**Вывод:** По метрикам виден размер базы и охват периода. Суммарная месячная выручка (MRR) задаёт масштаб бизнеса. Распределение по Basic / Standard / Premium показывает структуру тарифов и может указывать на потенциал upsell.

## Возраст пользователей и возрастные сегменты

Гистограмма и boxplot возраста, разбивка на возрастные группы и bar‑диаграмма по группам.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["age"], bins=25, ax=axes[0], color="coral")
axes[0].set_title("Распределение возраста пользователей")
axes[0].set_xlabel("Возраст")
axes[0].set_ylabel("Количество пользователей")
sns.boxplot(x=df["age"], ax=axes[1], color="salmon", showfliers=False)
axes[1].set_title("Возраст (ящик с усами)")
axes[1].set_xlabel("Возраст")
plt.tight_layout()
plt.show()

In [ ]:
# Возрастные корзины
bins = [18, 25, 35, 45, 55, 70]
labels = ["18–24", "25–34", "35–44", "45–54", "55–69"]
df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels, right=False)

age_counts = df["age_group"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 4))
age_counts.plot(kind="bar", ax=ax, color="coral", edgecolor="white")
ax.set_title("Количество пользователей по возрастным группам")
ax.set_xlabel("Возрастная группа")
ax.set_ylabel("Количество пользователей")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

**Вывод:** Доминирующие возрастные группы (часто 25–44 года) задают ядро аудитории. Это важно для позиционирования контента и таргетирования рекламы и upsell.

## Гендерный профиль аудитории

Распределение пользователей по полу.

In [ ]:
gender_counts = df["gender"].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
gender_counts.plot(kind="bar", ax=ax, color=["coral", "salmon"], edgecolor="white")
ax.set_title("Распределение пользователей по полу")
ax.set_xlabel("Пол")
ax.set_ylabel("Количество пользователей")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()
print(gender_counts.to_string())

**Вывод:** Соотношение мужчин и женщин характеризует гендерный профиль аудитории; при сильном перекосе имеет смысл учитывать это в контенте и коммуникациях.

## Устройства (девайсы) и способ потребления контента

Какие устройства используют подписчики — важно для приоритета платформ (мобильное приложение, TV, веб).

In [ ]:
device_counts = df["device"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
device_counts.plot(kind="bar", ax=ax, color="coral", edgecolor="white")
ax.set_title("Распределение пользователей по типу устройства")
ax.set_xlabel("Устройство")
ax.set_ylabel("Количество пользователей")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Устройства по возрастным группам (доли)
cross = pd.crosstab(df["age_group"], df["device"], normalize="index") * 100
fig, ax = plt.subplots(figsize=(10, 5))
cross.plot(kind="bar", ax=ax, stacked=False, colormap="Reds", edgecolor="white")
ax.set_title("Доля устройств по возрастным группам (%)")
ax.set_xlabel("Возрастная группа")
ax.set_ylabel("Доля, %")
ax.legend(title="Устройство", bbox_to_anchor=(1.02, 1))
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

**Вывод:** Доминирующие устройства (Smartphone, Smart TV, Laptop, Tablet) задают приоритеты для продукта: стабильность мобильного приложения, качество стриминга на TV и веб‑версии.

## Типы подписок и выручка

Распределение по тарифам (Basic / Standard / Premium), средняя выручка и суммарная выручка по типам.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

subs_count = df["subscription_type"].value_counts()
subs_count.plot(kind="bar", ax=axes[0], color=["coral", "salmon", "darkred"], edgecolor="white")
axes[0].set_title("Количество пользователей по типам подписки")
axes[0].set_xlabel("Тип подписки")
axes[0].set_ylabel("Количество пользователей")
axes[0].tick_params(axis="x", rotation=0)

rev_by_type = df.groupby("subscription_type")["monthly_revenue"].sum()
rev_by_type.plot(kind="bar", ax=axes[1], color=["coral", "salmon", "darkred"], edgecolor="white")
axes[1].set_title("Суммарная месячная выручка по типам подписки")
axes[1].set_xlabel("Тип подписки")
axes[1].set_ylabel("Сумма выручки")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

print("Средняя monthly_revenue по типам:")
print(df.groupby("subscription_type")["monthly_revenue"].agg(["mean", "sum"]).round(2).to_string())

**Вывод:** По количеству пользователей видно самые популярные тарифы; по сумме выручки — какой тариф даёт основной вклад в MRR. Перекос (мало Premium, но большой вклад в выручку) подчёркивает ценность upsell на премиум.

## География пользователей

Топ стран по количеству пользователей и по суммарной месячной выручке.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_countries = df["country"].value_counts().head(10)
top_countries.sort_values().plot(kind="barh", ax=axes[0], color="coral", edgecolor="white")
axes[0].set_title("Топ-10 стран по количеству пользователей")
axes[0].set_xlabel("Количество пользователей")
axes[0].set_ylabel("Страна")

rev_by_country = df.groupby("country")["monthly_revenue"].sum().nlargest(10)
rev_by_country.sort_values().plot(kind="barh", ax=axes[1], color="salmon", edgecolor="white")
axes[1].set_title("Топ-10 стран по суммарной месячной выручке")
axes[1].set_xlabel("Сумма выручки")
axes[1].set_ylabel("Страна")
plt.tight_layout()
plt.show()

**Вывод:** Ключевые рынки задают приоритеты локализации и маркетинга. Концентрация в нескольких странах или более равномерное распределение влияет на стратегию экспансии.

## Динамика привлечения пользователей

Число новых пользователей по месяцам (по join_date).

In [ ]:
df_join = df.set_index("join_date").resample("M")["user_id"].nunique()
fig, ax = plt.subplots(figsize=(10, 4))
df_join.plot(ax=ax, color="darkred", linewidth=2)
ax.set_title("Динамика привлечения пользователей по месяцам")
ax.set_xlabel("Месяц")
ax.set_ylabel("Количество новых пользователей")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Вывод:** Тренд (рост, плато или спад) и возможная сезонность помогают оценивать эффективность привлечения и планировать кампании.

## Динамика выручки и LTV‑подход

Оценка месяцев активности, накопленной выручки (LTV) по пользователю и распределение LTV по типам подписок.

In [ ]:
# Месяцы активности (не менее 1); единица M в timedelta не поддерживается — считаем через дни
delta = (df["last_payment_date"] - df["join_date"]).dt.days
months_active = (delta / 30.44).round().clip(lower=1)
df["months_active"] = months_active
df["total_revenue"] = df["monthly_revenue"] * df["months_active"]

print(f"Средний LTV (total_revenue): {df['total_revenue'].mean():.2f}")
print(f"Медианный LTV: {df['total_revenue'].median():.2f}")
print("\nСредний LTV по типам подписки:")
print(df.groupby("subscription_type")["total_revenue"].mean().round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["total_revenue"], bins=40, ax=axes[0], color="coral")
axes[0].set_title("Распределение накопленной выручки (LTV)")
axes[0].set_xlabel("Total revenue")
axes[0].set_ylabel("Количество пользователей")
sns.boxplot(data=df, x="subscription_type", y="total_revenue", ax=axes[1], palette="Reds")
axes[1].set_title("LTV по типам подписки")
axes[1].set_xlabel("Тип подписки")
axes[1].set_ylabel("Total revenue")
plt.tight_layout()
plt.show()

**Вывод:** Порядок величины LTV и его распределение по тарифам показывают, какие сегменты приносят больше выручки за время жизни. Premium даёт больший LTV при меньшей доле пользователей — основа для upsell.

## Связь тарифов с возрастом

Кто какой тариф выбирает: возрастные группы × тип подписки.

In [ ]:
cross_age = pd.crosstab(df["age_group"], df["subscription_type"])
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(cross_age, annot=True, fmt="d", cmap="Reds", ax=ax)
ax.set_title("Количество пользователей: возрастная группа × тип подписки")
ax.set_xlabel("Тип подписки")
ax.set_ylabel("Возрастная группа")
plt.tight_layout()
plt.show()

# Доли по строкам (внутри возраста — какой % на каком тарифе)
cross_pct = pd.crosstab(df["age_group"], df["subscription_type"], normalize="index") * 100
print("Доля типов подписки по возрастным группам (%):")
print(cross_pct.round(1).to_string())

**Вывод:** Видно, какие возрастные группы чаще выбирают Premium, Standard или Basic. Более платёжеспособные сегменты (часто 35–54) — кандидаты на таргетированный upsell.

## Связь тарифов со странами

Предпочтения тарифов по странам: где больше Premium, где Basic/Standard.

In [ ]:
# Топ-8 стран по числу пользователей
top8 = df["country"].value_counts().head(8).index
df_top = df[df["country"].isin(top8)]
cross_country = pd.crosstab(df_top["country"], df_top["subscription_type"])
cross_country_pct = cross_country.div(cross_country.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
cross_country_pct.plot(kind="bar", ax=ax, stacked=True, colormap="Reds", edgecolor="white")
ax.set_title("Доля типов подписки по странам (топ-8 по числу пользователей), %")
ax.set_xlabel("Страна")
ax.set_ylabel("Доля, %")
ax.legend(title="Тип подписки", bbox_to_anchor=(1.02, 1))
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

**Вывод:** В части стран выше доля Premium, в других — Basic/Standard. Это задаёт приоритеты по рынкам для премиум‑контента и ценовой политики.

## Итоговые бизнес‑выводы и рекомендации

**1. Профиль аудитории**
- Возраст: ядро аудитории — пользователи 25–44 лет; есть доля 18–24 и 45–54.
- Пол: соотношение мужчин и женщин задаёт гендерный профиль для контента и рекламы.
- Устройства: ключевые каналы потребления — смартфон, Smart TV и ноутбук; приоритет — стабильность мобильного и TV-приложений.
- География: топ стран по пользователям и выручке определяют ключевые рынки и приоритеты локализации.

**2. Подписки и выручка**
- По количеству пользователей лидирует один из тарифов (Basic/Standard/Premium); по выручке — вклад Premium при меньшей доле пользователей подчёркивает ценность премиум‑сегмента.
- LTV по типам подписок показывает, какие тарифы дают большую накопленную выручку; на этой основе можно оценивать эффективность upsell.

**3. Ключевые сегменты**
- Возрастные группы 35–44 и 45–54 чаще выбирают более дорогие тарифы — приоритет для таргетированного upsell.
- Страны с высокой долей Premium — целевые рынки для премиум‑контента и ценовых тестов.

**4. Рекомендации**
- **Upsell:** таргетировать переход на Premium на сегменты с высоким LTV и возрастом 35–54, а также на страны с уже высокой долей Standard.
- **Исследование:** углубить анализ по странам с низкой долей Premium (ценовая чувствительность, контент, промо).
- **Продукт:** усиливать опыт на доминирующих устройствах (смартфон, Smart TV); тестировать спец‑предложения и бандлы для сегментов с высоким потенциалом LTV.